# Case CIDACS — Notebook 2: Pipeline completo

Executa, em ordem, os **5 passos do `PLANO_SIMPLES.md`**:

| Passo | O que faz | Onde |
|---|---|---|
| 1 | Olhar os dados | seção 1 (resumo; detalhe no Notebook 1) |
| 2 | **Limpar os endereços** — a parte que mais importa | seção 2 → gera `src/laceda_setor/normalize.py` |
| 3 | Procurar o setor (cascata de 6 níveis) | seção 3 → gera `src/laceda_setor/match.py` |
| 4 | Gerar a saída | seção 4 → `output/resultado_final.csv` |
| 5 | Empacotar (README, testes, Docker) | seção 5 |

O notebook **escreve os módulos em `src/laceda_setor/`** (um por responsabilidade) com `%%writefile` e depois os importa. Assim o
código vive em arquivos versionáveis e testáveis, e o notebook continua sendo a narrativa
executável de ponta a ponta. O `src/main.py` da seção 5 reproduz o mesmo resultado com um
único comando, sem notebook.

**Entrega:** as 100 linhas originais intactas + `setor_censitario_encontrado`, com os casos
não localizados claramente identificados.

## 0. Ambiente

Prepara a sessão e fixa a raiz do projeto como diretório de trabalho, para que todos os
caminhos daqui em diante sejam relativos a ela. Também coloca `src/` no caminho de importação,
onde vive o pacote `laceda_setor`.

In [1]:
import os
import sys
import time
from pathlib import Path

import duckdb
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)                      # daqui em diante, todos os caminhos são relativos à raiz
sys.path.insert(0, "src")      # os módulos vivem em src/laceda_setor/

for pasta in ["output", "execucao-NB", "src/laceda_setor", "tests"]:
    Path(pasta).mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
con = duckdb.connect()
print("raiz:", RAIZ)

raiz: /home/carol/Área de trabalho/[LACEDA]Case1


---
# Passo 1 — Olhar os dados

Detalhado no Notebook 1. O que ficou fixado como regra obrigatória:

| # | Achado | Regra |
|---|---|---|
| 1 | `COD_SETOR` tem 16 chars terminando em `P` | cortar em 15 |
| 2 | Município: 7 dígitos no CNEFE, 6 na busca | juntar por 6 |
| 3 | Logradouro partido em 3 colunas no CNEFE | compor antes de comparar |
| 4 | 29% do CNEFE sem número | cascata não pode depender do número |
| 5 | CEP abrange de 1 a 38 setores | busca em cascata, não regra única |
| 6 | Zeros à esquerda significativos | ler tudo como string |

### 1.1 A base de referência

O CNEFE da Bahia é um CSV de 1,46 GB com separador `;` e 34 colunas. O pipeline **lê o arquivo
original direto**, sem converter para formato nenhum: a varredura acontece uma vez por
execução e já devolve só o recorte que interessa.

(Converter para Parquet economizaria ~9 s por execução e custaria ~12 s de conversão — não se
paga. O notebook de EDA usa um cache porque lá a mesma tabela é varrida dezenas de vezes.)

In [2]:
CNEFE_CSV = Path("data/raw/29_BA.csv")
assert CNEFE_CSV.exists(), f"baixe o CNEFE da Bahia para {CNEFE_CSV} (veja o README)"

# as 12 colunas úteis ao pareamento, das 34 do arquivo
COLUNAS_UTEIS = [
    "COD_UNICO_ENDERECO", "COD_MUNICIPIO", "COD_SETOR", "CEP", "DSC_LOCALIDADE",
    "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR", "NUM_ENDERECO",
    "DSC_MODIFICADOR", "LATITUDE", "LONGITUDE",
]

# all_varchar=true é obrigatório: zeros à esquerda de CEP, setor e município são significativos
FONTE_CNEFE = (f"read_csv_auto('{CNEFE_CSV}', delim=';', header=true, all_varchar=true)")

print(f"{CNEFE_CSV.stat().st_size / 1e9:.2f} GB")

1.46 GB


### 1.2 Carregar a base de busca

A base real (`data/raw/base_busca.*`) está no SharePoint da Fiocruz e exige login. Enquanto ela
não existir, o notebook gera uma **base sintética** a partir do próprio CNEFE, com sujeira
injetada de propósito (abreviações, acentos, CEP mascarado, `S/N`, bairro vazio).

A base sintética tem uma vantagem: como as linhas saem do CNEFE, **o setor correto é conhecido**
— dá para medir a acurácia da cascata na seção 4.3. Assim que `base_busca.xlsx` aparecer em
`data/raw/`, este notebook passa a usá-la automaticamente e a avaliação é pulada.

In [3]:
%%writefile src/laceda_setor/base_demo.py
"""
Gerador de uma base de busca SINTÉTICA, usada só enquanto a base real
(`data/raw/base_busca.xlsx`, 100 endereços, protegida por login no SharePoint)
não está disponível.

Serve para dois fins:
  - permitir rodar o pipeline de ponta a ponta antes de a base real chegar;
  - fornecer gabarito: como as linhas saem do próprio CNEFE, o setor correto é
    conhecido e dá para medir acurácia da cascata.

NÃO faz parte da entrega. Quando `base_busca.xlsx` existir, o notebook usa a
base real e ignora este módulo.
"""

from __future__ import annotations

import random

# sujeira injetada de propósito, espelhando o que o enunciado descreve
_ABREVIAR = {"RUA": "R.", "AVENIDA": "AV", "RODOVIA": "ROD", "PRACA": "PÇA",
             "TRAVESSA": "TV", "ESTRADA": "EST", "DOUTOR": "DR",
             "SANTA": "STA", "SANTO": "STO", "FAZENDA": "FAZ",
             "POVOADO": "POV", "NOSSA SENHORA": "N SRA"}
_ACENTOS = {"A": "Á", "E": "É", "I": "Í", "O": "Ó", "U": "Ú", "C": "Ç"}


def _abreviar(texto, rng):
    for cheio, curto in _ABREVIAR.items():
        if texto.startswith(cheio + " ") and rng.random() < 0.8:
            return curto + texto[len(cheio):]
    return texto


def _acentuar(texto, rng):
    saida = []
    trocou = False
    for ch in texto:
        if not trocou and ch in _ACENTOS and rng.random() < 0.15:
            saida.append(_ACENTOS[ch])
            trocou = True
        else:
            saida.append(ch)
    return "".join(saida)


def _errar_letra(texto, rng):
    if len(texto) < 6:
        return texto
    i = rng.randrange(3, len(texto) - 1)
    return texto[:i] + texto[i + 1:]       # some uma letra


def gerar(df_cnefe, n=100, semente=42):
    """Sorteia `n` endereços do CNEFE e devolve um DataFrame sujo + gabarito."""
    import pandas as pd

    rng = random.Random(semente)
    amostra = df_cnefe.sample(n=n, random_state=semente).reset_index(drop=True)

    linhas = []
    for i, r in amostra.iterrows():
        logr = r["n_logradouro"]
        dado = rng.random()
        if dado < 0.35:
            logr = _abreviar(logr, rng)
        if dado > 0.75:
            logr = _acentuar(logr, rng)
        if 0.90 < dado:
            logr = _errar_letra(logr, rng)
        if rng.random() < 0.4:
            logr = logr.title()

        cep = r["n_cep"]
        d = rng.random()
        if d < 0.25 and cep:
            cep = f"{cep[:5]}-{cep[5:]}"        # com máscara
        elif d < 0.35 and cep:
            cep = str(int(cep))                  # zero à esquerda comido pelo Excel
        elif d < 0.42:
            cep = ""                             # faltando

        num = r["n_numero"]
        if num == "SN":
            num = rng.choice(["S/N", "SN", "", "s/n"])

        bairro = r["n_bairro"]
        if rng.random() < 0.3:
            bairro = ""

        linhas.append({
            "id_consulta": i + 1,
            "consulta_municipio": r["n_municipio"],       # 6 dígitos, como no enunciado
            "consulta_cep": cep,
            "consulta_logradouro": logr,
            "consulta_numero": num,
            "consulta_bairro": bairro,
            "_gabarito_setor": r["setor_15"],             # só para avaliação
        })
    return pd.DataFrame(linhas)

Overwriting src/laceda_setor/base_demo.py


In [4]:
def localizar_base_busca():
    for p in sorted(Path("data/raw").glob("*")):
        if p.suffix.lower() in {".xlsx", ".xls", ".csv"} and "29_BA" not in p.name:
            return p
    return None

ARQUIVO_BUSCA = localizar_base_busca()
USANDO_BASE_REAL = ARQUIVO_BUSCA is not None

if USANDO_BASE_REAL:
    busca_bruta = (pd.read_excel(ARQUIVO_BUSCA, dtype=str)
                   if ARQUIVO_BUSCA.suffix.lower() != ".csv"
                   else pd.read_csv(ARQUIVO_BUSCA, dtype=str, sep=None, engine="python"))
    print(f"✅ base REAL: {ARQUIVO_BUSCA.name} — {len(busca_bruta)} linhas")
else:
    print("⚠️  base real ausente — será gerada uma base sintética na seção 1.4")
    busca_bruta = None

⚠️  base real ausente — será gerada uma base sintética na seção 1.4


### 1.3 Mapear as colunas da base de busca

O enunciado mostra `consulta_municipio`, `consulta_cep` e `consulta_logradouro`, mas os nomes
exatos só se confirmam com o arquivo em mãos. A detecção abaixo é por palavra-chave, para que o
pipeline não quebre se os nomes vierem diferentes.

In [5]:
def mapear_colunas(df):
    """Associa papel -> nome real da coluna, por palavra-chave."""
    chaves = {
        "municipio":  ["munic", "cod_mun", "ibge"],
        "cep":        ["cep"],
        "logradouro": ["logr", "endereco", "rua", "address"],
        "numero":     ["numero", "num", "nro"],
        "bairro":     ["bairro", "localidade", "distrito"],
    }
    mapa = {}
    for papel, termos in chaves.items():
        for coluna in df.columns:
            nome = coluna.lower()
            if any(t in nome for t in termos) and coluna not in mapa.values():
                mapa[papel] = coluna
                break
    return mapa

if USANDO_BASE_REAL:
    COLUNAS = mapear_colunas(busca_bruta)
    print(COLUNAS)
    faltando = {"municipio", "cep", "logradouro"} - set(COLUNAS)
    assert not faltando, f"colunas essenciais não detectadas: {faltando} — ajuste à mão"

### 1.4 Recorte do CNEFE: só os municípios que aparecem na busca

Descobre quais municípios a base de busca envolve e carrega do CNEFE apenas esses. É o
*blocking*: em vez de comparar 100 endereços contra 9 milhões, comparamos contra um recorte
pequeno — o que torna o problema resolvível em pandas puro.

In [6]:
from laceda_setor import normalize as nz   # será (re)gerado na seção 2; aqui serve para o recorte

def municipios_da_busca(df, coluna):
    return sorted({nz.normalizar_municipio(v) for v in df[coluna]} - {""})

if USANDO_BASE_REAL:
    MUNICIPIOS = municipios_da_busca(busca_bruta, COLUNAS["municipio"])
else:
    # sem a base real: 3 municípios de perfis diferentes (capital, interior urbano, rural)
    MUNICIPIOS = ["290070", "291720", "292740"]

print(f"{len(MUNICIPIOS)} municípios: {MUNICIPIOS}")

lista = ", ".join(f"'{m}'" for m in MUNICIPIOS)
t = time.time()
cnefe = con.execute(f'''
    SELECT {", ".join(COLUNAS_UTEIS)}
    FROM {FONTE_CNEFE}
    WHERE LEFT(COD_MUNICIPIO, 6) IN ({lista})
''').df()
print(f"recorte: {len(cnefe):,} endereços em {time.time() - t:.1f}s "
      f"({100 * len(cnefe) / 9_047_296:.1f}% do CNEFE da Bahia)")

3 municípios: ['290070', '291720', '292740']


recorte: 1,490,975 endereços em 15.6s (16.5% do CNEFE da Bahia)


Esse recorte é o *blocking*: em vez de comparar 100 endereços contra 9 milhões, comparamos
contra os poucos municípios que de fato aparecem na busca. É o que torna o problema resolvível
em pandas puro, sem Spark nem Elasticsearch.

---
# Passo 2 — Limpar os endereços

**A parte que mais importa.** As mesmas funções são aplicadas aos dois lados. Limpar de formas
diferentes garante que o match não aconteça.

A célula abaixo escreve `src/laceda_setor/normalize.py` — o módulo é o artefato da entrega; o notebook só o
gera e exercita.

In [7]:
%%writefile src/laceda_setor/normalize.py
"""
Normalização de endereços — Passo 2 do plano de trabalho.

Regra de ouro: as MESMAS funções são aplicadas aos dois lados (CNEFE e base de
busca). Se os lados forem limpos de formas diferentes, o match não acontece.

Decisões ancoradas no reconhecimento do Passo 1 (ver ACHADOS_PASSO1.md):
  - COD_SETOR do CNEFE tem 16 chars e termina em 'P'  -> cortar em 15.
  - COD_MUNICIPIO do CNEFE tem 7 dígitos; a base de busca usa 6 -> juntar por 6.
  - O logradouro do CNEFE vem partido em 3 colunas -> compor antes de comparar.
  - 29% do CNEFE não tem número (NUM_ENDERECO='0' + DSC_MODIFICADOR='SN').
"""

from __future__ import annotations

import re
import unicodedata

# --------------------------------------------------------------------------
# Dicionário de abreviações
# --------------------------------------------------------------------------
# Tipos de logradouro. Inclui os tipos RURAIS (FAZENDA, POVOADO, SITIO), que
# aparecem em mais de 1 milhão de linhas do CNEFE da Bahia.
TIPOS_LOGRADOURO = {
    "R": "RUA", "RU": "RUA",
    "AV": "AVENIDA", "AVEN": "AVENIDA", "AVN": "AVENIDA",
    "ROD": "RODOVIA", "RODV": "RODOVIA",
    "PCA": "PRACA", "PC": "PRACA", "PRC": "PRACA",
    "TV": "TRAVESSA", "TRAV": "TRAVESSA", "TRV": "TRAVESSA",
    "EST": "ESTRADA", "ESTR": "ESTRADA",
    "AL": "ALAMEDA", "ALM": "ALAMEDA",
    "LGO": "LARGO", "LG": "LARGO",
    "BC": "BECO",
    "CJ": "CONJUNTO", "CONJ": "CONJUNTO",
    "LOT": "LOTEAMENTO", "LOTE": "LOTEAMENTO",
    "VL": "VILA",
    "JD": "JARDIM", "JRD": "JARDIM",
    "FAZ": "FAZENDA",
    "POV": "POVOADO", "POVO": "POVOADO",
    "SIT": "SITIO",
    "CAM": "CAMINHO",
    "LAD": "LADEIRA",
    "PQ": "PARQUE",
    "DIST": "DISTRITO",
}

# Títulos e patentes que costumam vir abreviados no meio do nome.
TITULOS = {
    "DR": "DOUTOR", "DRA": "DOUTORA",
    "STA": "SANTA", "STO": "SANTO",
    "PE": "PADRE", "FR": "FREI",
    "PROF": "PROFESSOR", "PROFA": "PROFESSORA",
    "PRES": "PRESIDENTE",
    "GEN": "GENERAL", "CEL": "CORONEL", "MAL": "MARECHAL",
    "CAP": "CAPITAO", "TEN": "TENENTE", "SGT": "SARGENTO",
    "ENG": "ENGENHEIRO",
    "VER": "VEREADOR",
    "DEP": "DEPUTADO", "SEN": "SENADOR", "GOV": "GOVERNADOR",
    "MIN": "MINISTRO", "DES": "DESEMBARGADOR",
    "CMTE": "COMANDANTE",
    "MONS": "MONSENHOR",
}

ABREVIACOES = {**TIPOS_LOGRADOURO, **TITULOS}

# Sequências de mais de um token (aplicadas antes do passo token-a-token).
EXPRESSOES = [
    (r"\bN\s+SRA\b", "NOSSA SENHORA"),
    (r"\bN\s+S\b", "NOSSA SENHORA"),
    (r"\bNSA\s+SRA\b", "NOSSA SENHORA"),
    (r"\bNOSSA\s+SRA\b", "NOSSA SENHORA"),
    (r"\bSRA\s+DE\b", "SENHORA DE"),
    (r"\bS\s+SEBASTIAO\b", "SAO SEBASTIAO"),
    (r"\bSAO\s+JOAO\s+B\b", "SAO JOAO BATISTA"),
]

_MARCAS_SEM_NUMERO = {"", "SN", "S/N", "S N", "SN.", "S/N.", "0",
                      "SEM NUMERO", "SEMNUMERO", "S/NUMERO", "NAO INFORMADO"}

_NAO_ALFANUM = re.compile(r"[^A-Z0-9]+")
_ESPACOS = re.compile(r"\s+")
_DIGITOS = re.compile(r"\d+")


# --------------------------------------------------------------------------
# Blocos elementares
# --------------------------------------------------------------------------
def tirar_acentos(texto: str) -> str:
    """'ANDARAÍ' -> 'ANDARAI'."""
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def normalizar_texto(valor) -> str:
    """Maiúsculas, sem acento, sem pontuação, espaço simples."""
    if valor is None:
        return ""
    texto = str(valor).strip()
    if texto.lower() in {"nan", "none", "<na>", "null"}:
        return ""
    texto = tirar_acentos(texto).upper()
    texto = _NAO_ALFANUM.sub(" ", texto)
    return _ESPACOS.sub(" ", texto).strip()


def expandir_abreviacoes(texto: str) -> str:
    """'AV DR ABELARDO' -> 'AVENIDA DOUTOR ABELARDO'. Espera texto já normalizado."""
    if not texto:
        return ""
    for padrao, troca in EXPRESSOES:
        texto = re.sub(padrao, troca, texto)
    tokens = [ABREVIACOES.get(t, t) for t in texto.split()]
    return " ".join(tokens)


# --------------------------------------------------------------------------
# Campos do endereço
# --------------------------------------------------------------------------
def normalizar_logradouro(valor) -> str:
    """Pipeline completo de logradouro: texto limpo + abreviações expandidas."""
    return expandir_abreviacoes(normalizar_texto(valor))


def compor_logradouro_cnefe(tipo, titulo, nome) -> str:
    """
    O CNEFE guarda o logradouro em 3 colunas; a base de busca traz tudo junto.
    'AVENIDA' + 'DOUTOR' + 'ABELARDO VELOSO' -> 'AVENIDA DOUTOR ABELARDO VELOSO'.
    """
    partes = [normalizar_texto(p) for p in (tipo, titulo, nome)]
    return expandir_abreviacoes(" ".join(p for p in partes if p))


def normalizar_cep(valor) -> str:
    """Só dígitos, 8 posições com zero à esquerda. CEP inválido vira ''."""
    if valor is None:
        return ""
    digitos = "".join(_DIGITOS.findall(str(valor)))
    if not digitos or len(digitos) > 8:
        return ""
    cep = digitos.zfill(8)
    return "" if cep == "00000000" else cep


def normalizar_numero(valor) -> str:
    """
    'S/N', 'SN', vazio e '0' colapsam para o literal 'SN' — o mesmo que o CNEFE
    usa em 29% das linhas. '123-A' vira '123' (o sufixo não ajuda no match).
    """
    texto = normalizar_texto(valor)
    if texto in _MARCAS_SEM_NUMERO:
        return "SN"
    digitos = _DIGITOS.findall(texto)
    if not digitos:
        return "SN"
    numero = digitos[0].lstrip("0")
    return numero if numero else "SN"


def normalizar_municipio(valor) -> str:
    """
    Reduz a 6 dígitos — o denominador comum entre as duas bases.
    CNEFE '2900702' (7 dígitos, com DV) e busca '290070' (6) viram '290070'.
    """
    if valor is None:
        return ""
    digitos = "".join(_DIGITOS.findall(str(valor)))
    if len(digitos) >= 7:
        return digitos[:6]
    return digitos.zfill(6) if digitos else ""


def normalizar_setor(valor) -> str:
    """'290070220000001P' -> '290070220000001' (15 dígitos, formato do enunciado)."""
    if valor is None:
        return ""
    texto = str(valor).strip().upper()
    if texto.endswith("P"):
        texto = texto[:-1]
    return texto if texto.isdigit() and len(texto) == 15 else texto


def normalizar_bairro(valor) -> str:
    """Bairro/localidade: mesmo tratamento do logradouro, sem tipo de via."""
    return expandir_abreviacoes(normalizar_texto(valor))


# --------------------------------------------------------------------------
# Aplicação em lote (pandas)
# --------------------------------------------------------------------------
def normalizar_base_busca(df, colunas: dict):
    """
    Acrescenta ao DataFrame da base de busca as colunas `n_*` normalizadas.
    `colunas` mapeia papel -> nome real da coluna, ex.:
        {'municipio': 'consulta_municipio', 'cep': 'consulta_cep', ...}
    """
    out = df.copy()
    out["n_municipio"] = out[colunas["municipio"]].map(normalizar_municipio)
    out["n_cep"] = out[colunas["cep"]].map(normalizar_cep)
    out["n_logradouro"] = out[colunas["logradouro"]].map(normalizar_logradouro)
    out["n_numero"] = (out[colunas["numero"]].map(normalizar_numero)
                       if colunas.get("numero") else "SN")
    out["n_bairro"] = (out[colunas["bairro"]].map(normalizar_bairro)
                       if colunas.get("bairro") else "")
    return out


def aplicar_em_unicos(serie, funcao):
    """
    Aplica `funcao` só aos valores distintos e repropaga via `map`.

    O CNEFE tem milhões de linhas mas poucos valores distintos por coluna
    (ex.: ~30 mil logradouros em um recorte de 1,5 milhão de endereços).
    Normalizar o distinto e repropagar é uma ordem de grandeza mais rápido.
    """
    unicos = serie.astype("string").fillna("").unique()
    tabela = {v: funcao(v) for v in unicos}
    return serie.astype("string").fillna("").map(tabela)


def normalizar_cnefe(df):
    """Acrescenta as colunas `n_*` e `setor_15` ao recorte do CNEFE."""
    out = df.copy()
    out["n_municipio"] = aplicar_em_unicos(out["COD_MUNICIPIO"], normalizar_municipio)
    out["setor_15"] = aplicar_em_unicos(out["COD_SETOR"], normalizar_setor)
    out["n_cep"] = aplicar_em_unicos(out["CEP"], normalizar_cep)
    out["n_numero"] = aplicar_em_unicos(out["NUM_ENDERECO"], normalizar_numero)
    out["n_bairro"] = aplicar_em_unicos(out["DSC_LOCALIDADE"], normalizar_bairro)

    # logradouro depende de 3 colunas: deduplica a tripla antes de compor
    tipo = out["NOM_TIPO_SEGLOGR"].astype("string").fillna("")
    titulo = out["NOM_TITULO_SEGLOGR"].astype("string").fillna("")
    nome = out["NOM_SEGLOGR"].astype("string").fillna("")
    chave = tipo + "\x1f" + titulo + "\x1f" + nome
    tabela = {k: compor_logradouro_cnefe(*k.split("\x1f")) for k in chave.unique()}
    out["n_logradouro"] = chave.map(tabela)
    return out

Overwriting src/laceda_setor/normalize.py


In [8]:
import importlib
from laceda_setor import normalize as nz
importlib.reload(nz)
print("normalize.py recarregado —", len(nz.ABREVIACOES), "abreviações no dicionário")

normalize.py recarregado — 59 abreviações no dicionário


### 2.1 Testar nos casos do próprio enunciado

Exercita a normalização nos exemplos que o enunciado cita e em casos vindos dos achados do
Passo 1. É a verificação de que as regras fazem o que prometem, antes de aplicá-las a milhões
de linhas.

In [9]:
casos = [
    ("R. Rio Baiano",            "RUA RIO BAIANO"),
    ("Rodovia BR 101",           "RODOVIA BR 101"),
    ("AV DR ABELARDO VELOSO",    "AVENIDA DOUTOR ABELARDO VELOSO"),
    ("Pça N Sra da Conceição",   "PRACA NOSSA SENHORA DA CONCEICAO"),
    ("estr. do coco",            "ESTRADA DO COCO"),
    ("Faz. Boa Vista",           "FAZENDA BOA VISTA"),
]
pd.DataFrame([
    {"entrada": e, "saída": nz.normalizar_logradouro(e), "esperado": x,
     "ok": "✅" if nz.normalizar_logradouro(e) == x else "❌"}
    for e, x in casos
])

,entrada,saída,esperado,ok
0,R. Rio Baiano,RUA RIO BAIANO,RUA RIO BAIANO,✅
1,Rodovia BR 101,RODOVIA BR 101,RODOVIA BR 101,✅
2,AV DR ABELARDO VELOSO,AVENIDA DOUTOR ABELARDO VELOSO,AVENIDA DOUTOR ABELARDO VELOSO,✅
3,Pça N Sra da Conceição,PRACA NOSSA SENHORA DA CONCEICAO,PRACA NOSSA SENHORA DA CONCEICAO,✅
4,estr. do coco,ESTRADA DO COCO,ESTRADA DO COCO,✅
5,Faz. Boa Vista,FAZENDA BOA VISTA,FAZENDA BOA VISTA,✅


In [10]:
pd.DataFrame([
    {"campo": "CEP",       "entrada": repr(v), "saída": repr(nz.normalizar_cep(v))}
    for v in ["48030-260", "48030260", 8030260, "", None, "4803026"]
] + [
    {"campo": "número",    "entrada": repr(v), "saída": repr(nz.normalizar_numero(v))}
    for v in ["S/N", "SN", "", 0, "123-A", "0045"]
] + [
    {"campo": "município", "entrada": repr(v), "saída": repr(nz.normalizar_municipio(v))}
    for v in ["2900702", "290070", 2900702]
] + [
    {"campo": "setor",     "entrada": repr(v), "saída": repr(nz.normalizar_setor(v))}
    for v in ["290070220000001P", "290070220000001"]
])

,campo,entrada,saída
0,CEP,'48030-260','48030260'
1,CEP,'48030260','48030260'
2,CEP,8030260,'08030260'
3,CEP,'',''
4,CEP,None,''
5,CEP,'4803026','04803026'
6,número,'S/N','SN'
7,número,'SN','SN'
8,número,'','SN'
9,número,0,'SN'


Repare no CEP `8030260`: o Excel come o zero à esquerda. A normalização devolve `08030260`,
recuperando o valor. Esse é exatamente o tipo de sujeira que o enunciado anuncia.

### 2.2 Normalizar o lado CNEFE

Aplica as mesmas regras à base de referência e mostra, lado a lado, a coluna original e a
normalizada. Ver os dois lados juntos é o que permite conferir que a limpeza não destruiu
informação nem inventou nada.

In [11]:
t = time.time()
cnefe = nz.normalizar_cnefe(cnefe)
print(f"{len(cnefe):,} linhas normalizadas em {time.time() - t:.1f}s")

cnefe[["COD_MUNICIPIO", "n_municipio", "COD_SETOR", "setor_15", "CEP", "n_cep",
       "NOM_TIPO_SEGLOGR", "NOM_SEGLOGR", "n_logradouro", "NUM_ENDERECO",
       "n_numero", "n_bairro"]].head(8)

1,490,975 linhas normalizadas em 1.9s


,COD_MUNICIPIO,n_municipio,COD_SETOR,setor_15,CEP,n_cep,NOM_TIPO_SEGLOGR,NOM_SEGLOGR,n_logradouro,NUM_ENDERECO,n_numero,n_bairro
0,2900702,290070,290070220000001P,290070220000001,48031000,48031000,RUA,SEVERINO CALDEIRA,RUA SEVERINO CALDEIRA,0,SN,RIACHO DA GUIA
1,2900702,290070,290070205000186P,290070205000186,48024899,48024899,FAZENDA,BURY,FAZENDA BURY,0,SN,FAZENDA JACARE
2,2900702,290070,290070205000070P,290070205000070,48009044,48009044,RUA,JOSE JOAQUIM LEAL,RUA JOSE JOAQUIM LEAL,346,346,KENNEDY
3,2900702,290070,290070205000070P,290070205000070,48009044,48009044,RUA,JOSE JOAQUIM LEAL,RUA JOSE JOAQUIM LEAL,0,SN,KENNEDY
4,2927408,292740,292740805180195P,292740805180195,41370210,41370210,RUA,OSEAS LOPES,RUA OSEAS LOPES,59,59,JARDIM NOVA ESPERANCA
5,2927408,292740,292740805220279P,292740805220279,41207495,41207495,RUA,GABRIEL MONTEIRO DE CASTRO,RUA GABRIEL MONTEIRO DE CASTRO,26,26,TANCREDO NEVES
6,2927408,292740,292740805220405P,292740805220405,41120450,41120450,TRAVESSA,1 GUILHERME PEDROSA,TRAVESSA 1 GUILHERME PEDROSA,0,SN,PERNAMBUES
7,2927408,292740,292740805180171P,292740805180171,41320565,41320565,RUA,28,RUA 28,4,4,CASTELO BRANCO


`normalizar_cnefe` aplica cada função apenas aos **valores distintos** da coluna e repropaga o
resultado. Em um recorte de 1,5 milhão de endereços há só ~30 mil logradouros diferentes — a
diferença é de ~55s para ~4s.

### 2.3 Gerar a base de busca (real ou sintética) e normalizá-la

Carrega a base de busca — a real, se existir, ou a sintética — e passa por ela exatamente as
mesmas funções usadas no CNEFE. É o ponto em que a regra de ouro se cumpre: limpar os dois
lados de formas diferentes é o que impediria qualquer match.

In [12]:
from laceda_setor import base_demo
importlib.reload(base_demo)

if not USANDO_BASE_REAL:
    busca_bruta = base_demo.gerar(cnefe, n=100, semente=42)
    COLUNAS = mapear_colunas(busca_bruta.drop(columns=["_gabarito_setor"]))
    print("base sintética gerada |", COLUNAS)

busca = nz.normalizar_base_busca(busca_bruta, COLUNAS)
busca[[COLUNAS["logradouro"], "n_logradouro", COLUNAS["cep"], "n_cep",
       COLUNAS.get("numero", "n_numero"), "n_numero", "n_municipio"]].head(12)

base sintética gerada | {'municipio': 'consulta_municipio', 'cep': 'consulta_cep', 'logradouro': 'consulta_logradouro', 'numero': 'consulta_numero', 'bairro': 'consulta_bairro'}


,consulta_logradouro,n_logradouro,consulta_cep,n_cep,consulta_numero,n_numero,n_municipio
0,Travessa Gran Bell,TRAVESSA GRAN BELL,40760374,40760374,362,362,292740
1,RUA NOVO HORIZONTE,RUA NOVO HORIZONTE,40290735,40290735,140,140,292740
2,Rua Bosque Das Flores,RUA BOSQUE DAS FLORES,41502-203,41502203,S/N,SN,292740
3,VIA O 07,VIA O 07,,,s/n,SN,292740
4,RÚA MONZA,RUA MONZA,40800856,40800856,3,3,292740
5,VILA NOVO IMBUI,VILA NOVO IMBUI,41720003,41720003,289,289,292740
6,TRAVESSA ANA MARIANE BITENCOURT,TRAVESSA ANA MARIANE BITENCOURT,40390573,40390573,344,344,292740
7,RUA 18 DE AGOSTO,RUA 18 DE AGOSTO,,,,SN,292740
8,RUA BANANEIRAS,RUA BANANEIRAS,40760660,40760660,S/N,SN,292740
9,R. Cachoeira Veu Da Noiva,RUA CACHOEIRA VEU DA NOIVA,41400-312,41400312,1,1,292740


### 2.4 Conferir o que o passo 2 entrega ao passo 3

Mede quanto de cada campo normalizado ficou preenchido no recorte. Campo vazio é nível de
cascata que não vai funcionar, então esta tabela antecipa quais regras terão pouco material
para trabalhar.

In [13]:
CHAVES = ["n_municipio", "n_cep", "n_logradouro", "n_numero", "n_bairro", "setor_15"]

print("cobertura dos campos normalizados no recorte do CNEFE:")
display(pd.DataFrame({
    "preenchidos": (cnefe[CHAVES] != "").sum(),
    "pct": ((cnefe[CHAVES] != "").mean() * 100).round(1),
}))

cobertura dos campos normalizados no recorte do CNEFE:


,preenchidos,pct
n_municipio,1490975,100.0
n_cep,1490975,100.0
n_logradouro,1490975,100.0
n_numero,1490975,100.0
n_bairro,1490975,100.0
setor_15,1490975,100.0


Nada é gravado aqui. O dado normalizado segue em memória para o passo 3 — são 1,5 milhão de
linhas, que cabem folgado, e uma camada intermediária em disco só criaria um arquivo que
ninguém lê de volta.

---
# Passo 3 — Procurar o setor

Cascata de 6 níveis, do mais específico ao mais tolerante, parando no primeiro que resolve.

| nível | chave | confiança |
|---|---|---|
| 1 | município + CEP + logradouro + número (exato) | ALTA |
| 2 | município + CEP + logradouro (exato) | ALTA |
| 3 | município + logradouro + número (exato) | MEDIA |
| 4 | município + CEP, logradouro parecido (`rapidfuzz`) | MEDIA |
| 5 | município + bairro, logradouro parecido | BAIXA |
| 6 | nada bateu | — |

Os dois cuidados que o plano destaca:

- **vários endereços parecidos apontando para o mesmo setor não são ambiguidade** — a resposta
  está resolvida, e isso salva vários dos 100 casos;
- **apontando para setores diferentes → `AMBIGUO`, setor vazio.** Melhor assumir que não sabe do
  que chutar errado. Os candidatos ficam registrados numa coluna, para auditoria.

In [14]:
%%writefile src/laceda_setor/match.py
"""
Busca em cascata do setor censitário — Passo 3 do plano de trabalho.

A ideia é tentar regras da mais específica para a mais tolerante, parando na
primeira que resolve. Cada resposta carrega a regra que a produziu, para que a
saída seja auditável.

| nível | chave de busca                                   | confiança |
|-------|--------------------------------------------------|-----------|
| 1     | município + CEP + logradouro + número (exato)    | ALTA      |
| 2     | município + CEP + logradouro (exato)             | ALTA      |
| 3     | município + logradouro + número (exato)          | MEDIA     |
| 4     | município + CEP, logradouro parecido (fuzzy)     | MEDIA     |
| 5     | município + bairro, logradouro parecido (fuzzy)  | BAIXA     |
| 6     | nada bateu                                       | —         |

Dois cuidados que o plano destaca:
  - vários endereços parecidos apontando para o MESMO setor não são
    ambiguidade: a resposta está resolvida;
  - apontando para setores DIFERENTES, a resposta é `AMBIGUO` e o setor fica
    vazio. Melhor assumir que não sabe do que chutar errado.
"""

from __future__ import annotations

from collections import defaultdict

from rapidfuzz import fuzz, process

LIMIAR_FUZZY_CEP = 88     # nível 4: o CEP já restringe muito, pode ser tolerante
LIMIAR_FUZZY_BAIRRO = 92  # nível 5: bloco mais largo, exige mais parecença

STATUS_ENCONTRADO = "ENCONTRADO"
STATUS_AMBIGUO = "AMBIGUO"
STATUS_NAO_ENCONTRADO = "NAO_ENCONTRADO"


class IndiceCNEFE:
    """Índices em memória sobre o recorte do CNEFE já normalizado."""

    def __init__(self, df):
        """`df` precisa ter: n_municipio, n_cep, n_logradouro, n_numero,
        n_bairro, setor_15."""
        self.exato_cep_num = defaultdict(set)   # nível 1
        self.exato_cep = defaultdict(set)       # nível 2
        self.exato_num = defaultdict(set)       # nível 3
        self.bloco_cep = defaultdict(lambda: defaultdict(set))     # nível 4
        self.bloco_bairro = defaultdict(lambda: defaultdict(set))  # nível 5

        colunas = ["n_municipio", "n_cep", "n_logradouro", "n_numero",
                   "n_bairro", "setor_15"]
        for mun, cep, logr, num, bairro, setor in df[colunas].itertuples(index=False):
            if not mun or not logr or not setor:
                continue
            if cep:
                self.exato_cep[(mun, cep, logr)].add(setor)
                self.bloco_cep[(mun, cep)][logr].add(setor)
                if num != "SN":
                    self.exato_cep_num[(mun, cep, logr, num)].add(setor)
            if num != "SN":
                self.exato_num[(mun, logr, num)].add(setor)
            if bairro:
                self.bloco_bairro[(mun, bairro)][logr].add(setor)

        # listas de logradouros por bloco, pré-calculadas para o rapidfuzz
        self._logr_cep = {k: list(v) for k, v in self.bloco_cep.items()}
        self._logr_bairro = {k: list(v) for k, v in self.bloco_bairro.items()}

    def __len__(self):
        return len(self.exato_cep)


def _resolver(setores, regra, score, confianca):
    """Traduz um conjunto de setores candidatos em uma resposta."""
    setores = {s for s in setores if s}
    if not setores:
        return None
    if len(setores) == 1:
        return {
            "setor_censitario_encontrado": next(iter(setores)),
            "status": STATUS_ENCONTRADO,
            "regra": regra,
            "score": score,
            "confianca": confianca,
            "n_setores_candidatos": 1,
            "setores_candidatos": next(iter(setores)),
        }
    return {
        "setor_censitario_encontrado": "",
        "status": STATUS_AMBIGUO,
        "regra": regra,
        "score": score,
        "confianca": confianca,
        "n_setores_candidatos": len(setores),
        "setores_candidatos": "|".join(sorted(setores)),
    }


def _fuzzy(alvo, candidatos, mapa, limiar):
    """Melhor logradouro acima do limiar; devolve (setores, score) ou (None, None)."""
    if not alvo or not candidatos:
        return None, None
    achado = process.extractOne(alvo, candidatos, scorer=fuzz.token_sort_ratio,
                                score_cutoff=limiar)
    if achado is None:
        return None, None
    logr, score, _ = achado
    return mapa[logr], round(float(score), 1)


def buscar_setor(endereco: dict, indice: IndiceCNEFE,
                 limiar_cep: float = LIMIAR_FUZZY_CEP,
                 limiar_bairro: float = LIMIAR_FUZZY_BAIRRO) -> dict:
    """Roda a cascata para um endereço normalizado e devolve a resposta."""
    mun = endereco.get("n_municipio", "")
    cep = endereco.get("n_cep", "")
    logr = endereco.get("n_logradouro", "")
    num = endereco.get("n_numero", "SN")
    bairro = endereco.get("n_bairro", "")

    nao_encontrado = {
        "setor_censitario_encontrado": "",
        "status": STATUS_NAO_ENCONTRADO,
        "regra": "6_NAO_ENCONTRADO",
        "score": None,
        "confianca": "",
        "n_setores_candidatos": 0,
        "setores_candidatos": "",
    }

    if not mun or not logr:
        nao_encontrado["regra"] = "0_CAMPOS_INSUFICIENTES"
        return nao_encontrado

    # nível 1 — município + CEP + logradouro + número
    if cep and num != "SN":
        r = _resolver(indice.exato_cep_num.get((mun, cep, logr, num), set()),
                      "1_EXATO_CEP_LOGR_NUM", 100.0, "ALTA")
        if r:
            return r

    # nível 2 — município + CEP + logradouro
    if cep:
        r = _resolver(indice.exato_cep.get((mun, cep, logr), set()),
                      "2_EXATO_CEP_LOGR", 100.0, "ALTA")
        if r:
            return r

    # nível 3 — município + logradouro + número (sem CEP)
    if num != "SN":
        r = _resolver(indice.exato_num.get((mun, logr, num), set()),
                      "3_EXATO_LOGR_NUM", 100.0, "MEDIA")
        if r:
            return r

    # nível 4 — dentro do CEP, logradouro parecido
    if cep:
        mapa = indice.bloco_cep.get((mun, cep))
        if mapa:
            setores, score = _fuzzy(logr, indice._logr_cep[(mun, cep)], mapa, limiar_cep)
            if setores:
                r = _resolver(setores, "4_FUZZY_CEP", score, "MEDIA")
                if r:
                    return r

    # nível 5 — dentro do bairro, logradouro parecido
    if bairro:
        mapa = indice.bloco_bairro.get((mun, bairro))
        if mapa:
            setores, score = _fuzzy(logr, indice._logr_bairro[(mun, bairro)], mapa,
                                    limiar_bairro)
            if setores:
                r = _resolver(setores, "5_FUZZY_BAIRRO", score, "BAIXA")
                if r:
                    return r

    # nível 6 — nada bateu
    return nao_encontrado


def buscar_lote(df, indice: IndiceCNEFE, **kwargs):
    """Aplica a cascata linha a linha e devolve a lista de respostas."""
    return [buscar_setor(linha, indice, **kwargs)
            for linha in df.to_dict(orient="records")]

Overwriting src/laceda_setor/match.py


In [15]:
from laceda_setor import match as mt
importlib.reload(mt)

t = time.time()
indice = mt.IndiceCNEFE(cnefe)
print(f"índice construído em {time.time() - t:.1f}s — {len(indice):,} chaves exatas (nível 2)")

índice construído em 10.3s — 27,194 chaves exatas (nível 2)


### 3.1 Rodar a cascata

Executa a busca para todos os endereços e junta as respostas às colunas normalizadas. Cada
linha sai com o setor encontrado, o status e a regra que a resolveu.

In [16]:
t = time.time()
respostas = pd.DataFrame(mt.buscar_lote(busca, indice))
print(f"{len(busca)} endereços resolvidos em {time.time() - t:.2f}s")

resultado = pd.concat([busca.reset_index(drop=True), respostas], axis=1)
resultado[["n_logradouro", "n_cep", "n_numero", "setor_censitario_encontrado",
           "status", "regra", "score"]].head(15)

100 endereços resolvidos em 0.00s


,n_logradouro,n_cep,n_numero,setor_censitario_encontrado,status,regra,score
0,TRAVESSA GRAN BELL,40760374,362,292740805130002,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0
1,RUA NOVO HORIZONTE,40290735,140,292740805070158,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0
2,RUA BOSQUE DAS FLORES,41502203,SN,,AMBIGUO,2_EXATO_CEP_LOGR,100.0
3,VIA O 07,,SN,292740805230084,ENCONTRADO,5_FUZZY_BAIRRO,100.0
4,RUA MONZA,40800856,3,292740805130174,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0
5,VILA NOVO IMBUI,41720003,289,292740805060347,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0
6,TRAVESSA ANA MARIANE BITENCOURT,40390573,344,292740805220748,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0
7,RUA 18 DE AGOSTO,,SN,,AMBIGUO,5_FUZZY_BAIRRO,100.0
8,RUA BANANEIRAS,40760660,SN,,AMBIGUO,2_EXATO_CEP_LOGR,100.0
9,RUA CACHOEIRA VEU DA NOIVA,41400312,1,292740805230232,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0


### 3.2 Como cada nível contribuiu

Cruza regra e status para mostrar de onde vieram as respostas. Diz se a solução está se
apoiando em correspondência exata ou dependendo de similaridade — e, portanto, quanta confiança
o resultado merece.

In [17]:
resumo = (resultado.groupby(["regra", "status"]).size()
          .rename("linhas").reset_index().sort_values("regra"))
display(resumo)

print("\nPor status:")
display(resultado["status"].value_counts().rename("linhas").to_frame()
        .assign(pct=lambda d: (100 * d["linhas"] / len(resultado)).round(1)))

,regra,status,linhas
0,1_EXATO_CEP_LOGR_NUM,AMBIGUO,7
1,1_EXATO_CEP_LOGR_NUM,ENCONTRADO,62
2,2_EXATO_CEP_LOGR,AMBIGUO,6
3,2_EXATO_CEP_LOGR,ENCONTRADO,4
4,3_EXATO_LOGR_NUM,AMBIGUO,2
5,3_EXATO_LOGR_NUM,ENCONTRADO,8
6,4_FUZZY_CEP,AMBIGUO,5
7,4_FUZZY_CEP,ENCONTRADO,1
8,5_FUZZY_BAIRRO,AMBIGUO,1
9,5_FUZZY_BAIRRO,ENCONTRADO,1



Por status:


,linhas,pct
status,,
ENCONTRADO,76,76.0
AMBIGUO,21,21.0
NAO_ENCONTRADO,3,3.0


A taxa de `AMBIGUO` depende fortemente de **onde** os endereços estão. Em Salvador (`292740`),
nomes de logradouro se repetem em bairros distintos e um mesmo CEP cobre vários setores, então
um endereço sem número cai com frequência em mais de um setor. Em municípios menores o mesmo
endereço costuma resolver no nível 1.

Esses casos não são perdidos: o setor fica vazio, mas `setores_candidatos` guarda tudo o que a
busca viu. Quem for auditar consegue retomar exatamente de onde a solução parou — e, se a
comissão preferir uma política de desempate, ela pode ser aplicada sobre essa coluna sem
reprocessar nada.

### 3.3 Inspecionar os casos difíceis

Abre os ambíguos e os não encontrados, um a um. São os casos que mais ensinam: o ambíguo
mostra onde a chave de busca não discrimina o bastante, e o não encontrado mostra o que a
cascata não alcança.

In [18]:
print("— AMBÍGUOS (setores diferentes para o mesmo endereço) —")
amb = resultado[resultado["status"] == mt.STATUS_AMBIGUO]
display(amb[["n_logradouro", "n_cep", "n_numero", "regra",
             "n_setores_candidatos", "setores_candidatos"]].head(10))

print("— NÃO ENCONTRADOS —")
nao = resultado[resultado["status"] == mt.STATUS_NAO_ENCONTRADO]
display(nao[["n_municipio", "n_logradouro", "n_cep", "n_numero", "n_bairro", "regra"]].head(10))

— AMBÍGUOS (setores diferentes para o mesmo endereço) —


,n_logradouro,n_cep,n_numero,regra,n_setores_candidatos,setores_candidatos
2,RUA BOSQUE DAS FLORES,41502203,SN,2_EXATO_CEP_LOGR,2,292740805230168|292740805230181
7,RUA 18 DE AGOSTO,,SN,5_FUZZY_BAIRRO,3,292740805070117|292740805070168|292740805070255
8,RUA BANANEIRAS,40760660,SN,2_EXATO_CEP_LOGR,3,292740805130002|292740805130064|292740805130137
14,AVENIDA EDGARD SANTOS,41192005,SN,2_EXATO_CEP_LOGR,22,292740805220234|292740805220242|29274080522024...
18,TRAVESSA BLA VISTA,41615020,37,4_FUZZY_CEP,2,292740805090184|292740805090330
20,ESTRADA DA PACIENCIA,41338700,10,1_EXATO_CEP_LOGR_NUM,4,292740805260047|292740805260186|29274080526018...
21,RUA GOLAN,40750270,199,1_EXATO_CEP_LOGR_NUM,2,292740805130005|292740805130170
24,RUA SILVIO ARAUJO,40726320,SN,2_EXATO_CEP_LOGR,4,292740805160122|292740805160124|29274080516014...
29,RUA VIVALDO JESUINO DE SOUZA,41302505,SN,2_EXATO_CEP_LOGR,2,292740805260220|292740805260359
30,RUA ONZE DE AGOSTO,,11,3_EXATO_LOGR_NUM,3,292740805270315|292740805270337|292740805270338


— NÃO ENCONTRADOS —


,n_municipio,n_logradouro,n_cep,n_numero,n_bairro,regra
63,292740,RUA NOVAALIANCA,40725040,72,,6_NAO_ENCONTRADO
94,292740,RUA JOAQUIM DOS COUROS,,SN,,6_NAO_ENCONTRADO
98,292740,RUA RUL SEIXAS,41350450,SN,MUSSURUNGA,6_NAO_ENCONTRADO


### 3.4 Sensibilidade ao limiar do fuzzy

O limiar não é um número mágico — vale mostrar o efeito de mexer nele. Mais tolerante
(limiar baixo) encontra mais e erra mais; mais rígido encontra menos com mais segurança.

In [19]:
linhas = []
for limiar in [80, 85, 88, 92, 95, 100]:
    r = pd.DataFrame(mt.buscar_lote(busca, indice, limiar_cep=limiar, limiar_bairro=limiar))
    vc = r["status"].value_counts()
    linhas.append({
        "limiar": limiar,
        "encontrado": int(vc.get(mt.STATUS_ENCONTRADO, 0)),
        "ambiguo": int(vc.get(mt.STATUS_AMBIGUO, 0)),
        "nao_encontrado": int(vc.get(mt.STATUS_NAO_ENCONTRADO, 0)),
        "resolvidos_por_fuzzy": int(r["regra"].str.startswith(("4_", "5_")).sum()),
    })
pd.DataFrame(linhas)

,limiar,encontrado,ambiguo,nao_encontrado,resolvidos_por_fuzzy
0,80,76,22,2,9
1,85,76,21,3,8
2,88,76,21,3,8
3,92,76,21,3,8
4,95,76,20,4,7
5,100,75,16,9,2


---
# Passo 4 — Gerar a saída

`output/resultado_final.csv` com **as linhas originais intactas** + as colunas de resposta.
Nenhuma coluna de entrada é alterada ou removida — o avaliador precisa reconhecer a base dele.

In [20]:
COLUNAS_SAIDA = ["setor_censitario_encontrado", "status", "regra", "score",
                 "confianca", "n_setores_candidatos", "setores_candidatos"]

colunas_originais = [c for c in busca_bruta.columns if not c.startswith("_gabarito")]
final = pd.concat([busca_bruta[colunas_originais].reset_index(drop=True),
                   respostas[COLUNAS_SAIDA]], axis=1)

final.to_csv("output/resultado_final.csv", index=False, encoding="utf-8")
final.to_parquet("output/resultado_final.parquet", index=False)
try:
    final.to_excel("output/resultado_final.xlsx", index=False)
except Exception as e:
    print("xlsx não gerado:", e)

print(f"{len(final)} linhas × {len(final.columns)} colunas")
final.head(10)

100 linhas × 13 colunas


,id_consulta,consulta_municipio,consulta_cep,consulta_logradouro,consulta_numero,consulta_bairro,setor_censitario_encontrado,status,regra,score,confianca,n_setores_candidatos,setores_candidatos
0,1,292740,40760374,Travessa Gran Bell,362,,292740805130002,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0,ALTA,1,292740805130002
1,2,292740,40290735,RUA NOVO HORIZONTE,140,,292740805070158,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0,ALTA,1,292740805070158
2,3,292740,41502-203,Rua Bosque Das Flores,S/N,JARDIM DAS MARGARIDAS,,AMBIGUO,2_EXATO_CEP_LOGR,100.0,ALTA,2,292740805230168|292740805230181
3,4,292740,,VIA O 07,s/n,SAO CRISTOVAO,292740805230084,ENCONTRADO,5_FUZZY_BAIRRO,100.0,BAIXA,1,292740805230084
4,5,292740,40800856,RÚA MONZA,3,PARIPE,292740805130174,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0,ALTA,1,292740805130174
5,6,292740,41720003,VILA NOVO IMBUI,289,,292740805060347,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0,ALTA,1,292740805060347
6,7,292740,40390573,TRAVESSA ANA MARIANE BITENCOURT,344,SAO CAETANO,292740805220748,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0,ALTA,1,292740805220748
7,8,292740,,RUA 18 DE AGOSTO,,CANDEAL,,AMBIGUO,5_FUZZY_BAIRRO,100.0,BAIXA,3,292740805070117|292740805070168|292740805070255
8,9,292740,40760660,RUA BANANEIRAS,S/N,,,AMBIGUO,2_EXATO_CEP_LOGR,100.0,ALTA,3,292740805130002|292740805130064|292740805130137
9,10,292740,41400-312,R. Cachoeira Veu Da Noiva,1,,292740805230232,ENCONTRADO,1_EXATO_CEP_LOGR_NUM,100.0,ALTA,1,292740805230232


Significado das colunas acrescentadas:

| coluna | o que é |
|---|---|
| `setor_censitario_encontrado` | o código de 15 dígitos; **vazio** quando não resolvido |
| `status` | `ENCONTRADO` / `AMBIGUO` / `NAO_ENCONTRADO` — a identificação clara que o case exige |
| `regra` | qual dos 6 níveis resolveu (auditoria) |
| `score` | similaridade, quando houve fuzzy; 100 nos níveis exatos |
| `confianca` | ALTA / MEDIA / BAIXA, conforme o nível |
| `n_setores_candidatos`, `setores_candidatos` | o que a busca viu nos casos ambíguos |

### 4.1 Resumo da execução

Condensa o resultado nos números que o avaliador vai querer de imediato — totais por status,
taxa de resolução e divisão entre regra exata e fuzzy — e grava esses resumos em `output/`.

In [21]:
total = len(final)
vc = final["status"].value_counts()
resumo_exec = pd.DataFrame({
    "métrica": ["total de endereços", "ENCONTRADO", "AMBIGUO", "NAO_ENCONTRADO",
                "taxa de resolução", "resolvidos por regra exata", "resolvidos por fuzzy"],
    "valor": [
        total,
        int(vc.get("ENCONTRADO", 0)),
        int(vc.get("AMBIGUO", 0)),
        int(vc.get("NAO_ENCONTRADO", 0)),
        f"{100 * vc.get('ENCONTRADO', 0) / total:.1f}%",
        int(final["regra"].str.startswith(("1_", "2_", "3_")).sum()),
        int(final["regra"].str.startswith(("4_", "5_")).sum()),
    ],
})
display(resumo_exec)

resumo_exec.to_csv("output/resumo_execucao.csv", index=False)
(final.groupby(["regra", "status"]).size().rename("linhas").reset_index()
 .to_csv("output/resumo_por_regra.csv", index=False))
print("resumos salvos em output/")

,métrica,valor
0,total de endereços,100
1,ENCONTRADO,76
2,AMBIGUO,21
3,NAO_ENCONTRADO,3
4,taxa de resolução,76.0%
5,resolvidos por regra exata,89
6,resolvidos por fuzzy,8


resumos salvos em output/


### 4.2 Log de execução (entregável opcional do case)

Registra em JSON as condições exatas da execução: qual base foi usada, quais municípios,
quais limiares, que resultado deu. É o que permite a outra pessoa saber o que foi rodado sem
precisar perguntar.

In [22]:
import json
from datetime import datetime

log = {
    "executado_em": datetime.now().isoformat(timespec="seconds"),
    "base_busca": (ARQUIVO_BUSCA.name if USANDO_BASE_REAL else "SINTÉTICA (base real ausente)"),
    "usando_base_real": USANDO_BASE_REAL,
    "municipios_carregados": MUNICIPIOS,
    "enderecos_cnefe_no_recorte": int(len(cnefe)),
    "chaves_no_indice": len(indice),
    "limiar_fuzzy_cep": mt.LIMIAR_FUZZY_CEP,
    "limiar_fuzzy_bairro": mt.LIMIAR_FUZZY_BAIRRO,
    "status": {k: int(v) for k, v in final["status"].value_counts().items()},
    "regras": {k: int(v) for k, v in final["regra"].value_counts().items()},
}
Path("output/log_execucao.json").write_text(json.dumps(log, indent=2, ensure_ascii=False))
print(json.dumps(log, indent=2, ensure_ascii=False))

{
  "executado_em": "2026-10-13T07:48:38",
  "base_busca": "SINTÉTICA (base real ausente)",
  "usando_base_real": false,
  "municipios_carregados": [
    "290070",
    "291720",
    "292740"
  ],
  "enderecos_cnefe_no_recorte": 1490975,
  "chaves_no_indice": 27194,
  "limiar_fuzzy_cep": 88,
  "limiar_fuzzy_bairro": 92,
  "status": {
    "ENCONTRADO": 76,
    "AMBIGUO": 21,
    "NAO_ENCONTRADO": 3
  },
  "regras": {
    "1_EXATO_CEP_LOGR_NUM": 69,
    "2_EXATO_CEP_LOGR": 10,
    "3_EXATO_LOGR_NUM": 10,
    "4_FUZZY_CEP": 6,
    "6_NAO_ENCONTRADO": 3,
    "5_FUZZY_BAIRRO": 2
  }
}


### 4.3 Avaliação contra gabarito

Só roda com a base sintética, onde o setor correto é conhecido por construção. Com a base real
a comissão compara contra o padrão ouro dela, e esta seção é pulada.

As duas métricas que importam são diferentes:

- **precisão** — entre os que a solução afirmou ter encontrado, quantos estão certos. É o que
  mede se dá para confiar na resposta.
- **cobertura** — de quantos dos 100 a solução conseguiu responder.

Uma cascata honesta prefere perder cobertura a perder precisão: marcar `AMBIGUO` em vez de
chutar.

In [23]:
if not USANDO_BASE_REAL:
    aval = resultado.copy()
    aval["correto"] = aval["setor_censitario_encontrado"] == aval["_gabarito_setor"]
    enc = aval[aval["status"] == "ENCONTRADO"]
    amb = aval[aval["status"] == "AMBIGUO"]

    display(pd.DataFrame({
        "métrica": ["cobertura (ENCONTRADO / total)",
                    "precisão (corretos / ENCONTRADO)",
                    "acurácia global (corretos / total)",
                    "AMBIGUO cujo conjunto continha o setor certo"],
        "valor": [
            f"{100 * len(enc) / len(aval):.1f}%",
            f"{100 * enc['correto'].mean():.1f}%" if len(enc) else "—",
            f"{100 * aval['correto'].mean():.1f}%",
            f"{sum(g in str(c).split('|') for g, c in zip(amb['_gabarito_setor'], amb['setores_candidatos']))}/{len(amb)}",
        ],
    }))

    print("\nPrecisão por regra:")
    display(enc.groupby("regra")["correto"].agg(["size", "sum", "mean"])
            .rename(columns={"size": "linhas", "sum": "corretos", "mean": "precisão"}))
else:
    print("base real em uso — avaliação feita pela comissão contra o padrão ouro")

,métrica,valor
0,cobertura (ENCONTRADO / total),76.0%
1,precisão (corretos / ENCONTRADO),100.0%
2,acurácia global (corretos / total),76.0%
3,AMBIGUO cujo conjunto continha o setor certo,21/21



Precisão por regra:


,linhas,corretos,precisão
regra,,,
1_EXATO_CEP_LOGR_NUM,62,62,1.0
2_EXATO_CEP_LOGR,4,4,1.0
3_EXATO_LOGR_NUM,8,8,1.0
4_FUZZY_CEP,1,1,1.0
5_FUZZY_BAIRRO,1,1,1.0


Ler o resultado com honestidade: a base sintética é **otimista**, porque a sujeira foi injetada
por um gerador conhecido e todo endereço existe no CNEFE por construção. Na base real haverá
endereços que simplesmente não constam do cadastro, e a taxa de `NAO_ENCONTRADO` será maior.
O que a avaliação valida aqui é a **mecânica da cascata**, não a taxa de acerto esperada.

---
# Passo 5 — Empacotar

Objetivo: o avaliador clona, roda **um comando** e obtém o mesmo CSV.

### 5.1 `requirements.txt` com versões fixas

Gera o arquivo de dependências a partir das versões realmente instaladas nesta execução, em
vez de escrevê-las de memória. Versão fixa é o que faz a reprodução dar o mesmo resultado.

In [24]:
import duckdb as _d, pandas as _p, rapidfuzz as _r, pyarrow as _a
reqs = "\n".join([
    f"pandas=={_p.__version__}",
    f"duckdb=={_d.__version__}",
    f"rapidfuzz=={_r.__version__}",
    f"pyarrow=={_a.__version__}",
    "openpyxl>=3.1",
    "pytest>=8.0",
]) + "\n"
Path("requirements.txt").write_text(reqs)
print(reqs)

pandas==3.0.6
duckdb==1.5.6
rapidfuzz==3.14.6
pyarrow==26.0.0
openpyxl>=3.1
pytest>=8.0



### 5.2 Os módulos de execução — uma responsabilidade por arquivo

As seções 2 e 3 já escreveram `normalize.py` (limpar) e `match.py` (procurar). Falta o que
liga as duas pontas. Em vez de um `main.py` que faz tudo, cada arquivo abaixo tem um motivo
só para mudar:

| arquivo | responsabilidade |
|---|---|
| `config.py` | constantes e caminhos padrão |
| `leitura.py` | achar e ler o CNEFE e a base de busca |
| `colunas.py` | descobrir o esquema da base de busca |
| `pipeline.py` | a ordem dos passos, de ponta a ponta |
| `relatorio.py` | os números da execução |
| `escrita.py` | gravar resultado e log |
| `cli.py` | argumentos de linha de comando e impressão |

É o que torna a solução reproduzível por outra pessoa, requisito que vale 20% da avaliação.

#### As peças


In [25]:
%%writefile src/laceda_setor/config.py
"""
Constantes e caminhos padrão do pipeline. Nenhuma lógica — só os valores que
os outros módulos compartilham.
"""

from __future__ import annotations

from pathlib import Path

# raiz do projeto: .../[LACEDA]Case1
RAIZ = Path(__file__).resolve().parents[2]
DIR_DADOS = RAIZ / "data" / "raw"
DIR_SAIDA = RAIZ / "output"

CNEFE_PADRAO = DIR_DADOS / "29_BA.csv"
SAIDA_PADRAO = DIR_SAIDA / "resultado_final.csv"
NOME_LOG = "log_execucao.json"

# as 12 colunas do CNEFE que interessam ao pareamento (das 34 do arquivo)
COLUNAS_UTEIS = [
    "COD_UNICO_ENDERECO", "COD_MUNICIPIO", "COD_SETOR", "CEP", "DSC_LOCALIDADE",
    "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR", "NUM_ENDERECO",
    "DSC_MODIFICADOR", "LATITUDE", "LONGITUDE",
]

# papel do campo -> palavras-chave que o identificam na base de busca
PAPEIS = {
    "municipio": ["munic", "cod_mun", "ibge"],
    "cep": ["cep"],
    "logradouro": ["logr", "endereco", "rua", "address"],
    "numero": ["numero", "num", "nro"],
    "bairro": ["bairro", "localidade", "distrito"],
}
PAPEIS_OBRIGATORIOS = {"municipio", "cep", "logradouro"}

EXTENSOES_BUSCA = {".xlsx", ".xls", ".csv"}
PREFIXO_CNEFE = "29_BA"          # arquivo do CNEFE, nunca é a base de busca
PREFIXO_GABARITO = "_gabarito"   # colunas da base sintética, fora da entrega


Overwriting src/laceda_setor/config.py


In [26]:
%%writefile src/laceda_setor/colunas.py
"""
Descoberta do esquema da base de busca: associa cada papel do endereço
(município, CEP, logradouro, ...) ao nome real da coluna recebida.

Responsabilidade única: interpretar nomes de coluna. Não lê nem escreve nada.
"""

from __future__ import annotations

from .config import PAPEIS, PAPEIS_OBRIGATORIOS


def mapear_colunas(df, papeis: dict | None = None) -> dict:
    """Associa papel -> nome real da coluna da base de busca, por palavra-chave."""
    papeis = papeis or PAPEIS
    mapa: dict[str, str] = {}
    for papel, termos in papeis.items():
        for coluna in df.columns:
            if any(t in coluna.lower() for t in termos) and coluna not in mapa.values():
                mapa[papel] = coluna
                break
    return mapa


def papeis_faltando(mapa: dict) -> set:
    """Papéis essenciais que não foram encontrados no cabeçalho."""
    return PAPEIS_OBRIGATORIOS - set(mapa)


Overwriting src/laceda_setor/colunas.py


In [27]:
%%writefile src/laceda_setor/leitura.py
"""
Entrada de dados: achar e ler os arquivos. Responsabilidade única — nada aqui
normaliza, pareia ou escreve.
"""

from __future__ import annotations

from pathlib import Path

import pandas as pd

from .config import COLUNAS_UTEIS, DIR_DADOS, EXTENSOES_BUSCA, PREFIXO_CNEFE


def localizar_base_busca(diretorio: Path = DIR_DADOS) -> Path | None:
    """Primeiro arquivo tabular de `data/raw/` que não seja o CNEFE."""
    for caminho in sorted(diretorio.glob("*")):
        if caminho.suffix.lower() in EXTENSOES_BUSCA and PREFIXO_CNEFE not in caminho.name:
            return caminho
    return None


def ler_busca(caminho: Path) -> pd.DataFrame:
    """Lê a base de busca (.xlsx/.xls/.csv) inteira como texto."""
    if caminho.suffix.lower() == ".csv":
        return pd.read_csv(caminho, dtype=str, sep=None, engine="python")
    return pd.read_excel(caminho, dtype=str)


def carregar_cnefe(csv: Path, municipios: list[str], con) -> pd.DataFrame:
    """
    Lê do CSV só os municípios que aparecem na base de busca (blocking).

    O DuckDB varre o arquivo de 1,46 GB e devolve apenas o recorte (~13s). Não há
    camada intermediária em disco: a base original é lida uma vez por execução e o
    resto do pipeline acontece em memória.

    `all_varchar=true` é obrigatório — zeros à esquerda de CEP, setor e código de
    município são significativos.
    """
    if not Path(csv).exists():
        raise FileNotFoundError(f"CNEFE não encontrado em {csv} (veja o README)")
    lista = ", ".join(f"'{m}'" for m in municipios)
    return con.execute(f"""
        SELECT {', '.join(COLUNAS_UTEIS)}
        FROM read_csv_auto('{csv}', delim=';', header=true, all_varchar=true)
        WHERE LEFT(COD_MUNICIPIO, 6) IN ({lista})
    """).df()


Overwriting src/laceda_setor/leitura.py


In [28]:
%%writefile src/laceda_setor/escrita.py
"""
Saída de dados: gravar o resultado e o log de execução. Responsabilidade única —
recebe o que já foi calculado e só persiste em disco.
"""

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from .config import NOME_LOG


def salvar_resultado(df: pd.DataFrame, saida: Path) -> Path:
    """Grava o resultado em CSV e, ao lado, a mesma tabela em Parquet."""
    saida = Path(saida)
    saida.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(saida, index=False, encoding="utf-8")
    df.to_parquet(saida.with_suffix(".parquet"), index=False)
    return saida


def salvar_log(log: dict, diretorio: Path, nome: str = NOME_LOG) -> Path:
    """Grava o log da execução em JSON legível."""
    diretorio = Path(diretorio)
    diretorio.mkdir(parents=True, exist_ok=True)
    caminho = diretorio / nome
    caminho.write_text(json.dumps(log, indent=2, ensure_ascii=False))
    return caminho


Overwriting src/laceda_setor/escrita.py


In [29]:
%%writefile src/laceda_setor/relatorio.py
"""
Resumo da execução: transforma o resultado em números. Responsabilidade única —
só agrega e formata; não grava arquivos nem imprime.
"""

from __future__ import annotations

from datetime import datetime

import pandas as pd


def contar(serie: pd.Series) -> dict:
    """Contagem por valor, já convertida para tipos nativos do Python."""
    return {str(k): int(v) for k, v in serie.value_counts().items()}


def montar_log(final: pd.DataFrame, *, base_busca: str, municipios: list[str],
               enderecos_cnefe: int, segundos: float) -> dict:
    """Dicionário com tudo que descreve a execução, pronto para virar JSON."""
    return {
        "executado_em": datetime.now().isoformat(timespec="seconds"),
        "base_busca": base_busca,
        "municipios": municipios,
        "enderecos_cnefe_no_recorte": int(enderecos_cnefe),
        "status": contar(final["status"]),
        "regras": contar(final["regra"]),
        "segundos": round(segundos, 1),
    }


def linhas_resumo(final: pd.DataFrame) -> list[str]:
    """Linhas 'STATUS  n  (x%)' para exibição no terminal."""
    total = len(final) or 1
    return [f"  {chave:<16} {valor:>4}  ({100 * valor / total:.1f}%)"
            for chave, valor in contar(final["status"]).items()]


Overwriting src/laceda_setor/relatorio.py


#### A orquestração

`executar()` é só a ordem dos passos — nenhuma regra de endereço, SQL ou `argparse` mora aqui.


In [30]:
%%writefile src/laceda_setor/pipeline.py
"""
Orquestração do pipeline: encadeia leitura -> normalização -> cascata -> saída.

Responsabilidade única: a ordem dos passos. Cada passo em si mora no seu módulo;
aqui não há regra de endereço, SQL nem argparse.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from pathlib import Path

import duckdb
import pandas as pd

from . import colunas as cl
from . import escrita, leitura, relatorio
from . import match as mt
from . import normalize as nz
from .config import CNEFE_PADRAO, DIR_DADOS, PREFIXO_GABARITO, SAIDA_PADRAO


class ErroDePipeline(Exception):
    """Falha prevista de execução (arquivo ausente, esquema incompleto)."""


@dataclass
class Opcoes:
    """Tudo que o pipeline precisa saber para rodar."""
    cnefe: Path = CNEFE_PADRAO
    busca: Path | None = None
    saida: Path = SAIDA_PADRAO
    limiar_cep: float = mt.LIMIAR_FUZZY_CEP
    limiar_bairro: float = mt.LIMIAR_FUZZY_BAIRRO


@dataclass
class Resultado:
    """O que a execução produziu, para quem quiser exibir ou testar."""
    final: pd.DataFrame
    log: dict
    saida: Path
    colunas_detectadas: dict = field(default_factory=dict)


def resolver_base_busca(busca: Path | None, diretorio: Path = DIR_DADOS) -> Path:
    """Usa a base indicada ou procura uma em `data/raw/`."""
    caminho = Path(busca) if busca else leitura.localizar_base_busca(diretorio)
    if caminho is None:
        raise ErroDePipeline(f"base de busca não encontrada em {diretorio} (veja o README)")
    if not caminho.exists():
        raise ErroDePipeline(f"base de busca não encontrada: {caminho}")
    return caminho


def preparar_busca(bruta: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Detecta o esquema da base de busca e devolve a versão normalizada."""
    mapa = cl.mapear_colunas(bruta)
    faltando = cl.papeis_faltando(mapa)
    if faltando:
        raise ErroDePipeline(f"colunas essenciais não detectadas: {faltando}")
    return nz.normalizar_base_busca(bruta, mapa), mapa


def parear(busca: pd.DataFrame, cnefe: pd.DataFrame, opcoes: Opcoes) -> pd.DataFrame:
    """Monta o índice do CNEFE e roda a cascata para cada endereço."""
    indice = mt.IndiceCNEFE(nz.normalizar_cnefe(cnefe))
    return pd.DataFrame(mt.buscar_lote(busca, indice,
                                       limiar_cep=opcoes.limiar_cep,
                                       limiar_bairro=opcoes.limiar_bairro))


def juntar(bruta: pd.DataFrame, respostas: pd.DataFrame) -> pd.DataFrame:
    """Colunas originais da base de busca + resposta, lado a lado."""
    originais = [c for c in bruta.columns if not c.startswith(PREFIXO_GABARITO)]
    return pd.concat([bruta[originais].reset_index(drop=True), respostas], axis=1)


def executar(opcoes: Opcoes, aviso=lambda _: None) -> Resultado:
    """Roda o pipeline inteiro. `aviso` recebe mensagens de progresso."""
    inicio = time.time()

    caminho_busca = resolver_base_busca(opcoes.busca)
    bruta = leitura.ler_busca(caminho_busca)
    busca, mapa = preparar_busca(bruta)
    aviso(f"base de busca: {caminho_busca.name} ({len(bruta)} linhas) | colunas: {mapa}")

    municipios = sorted(set(busca["n_municipio"]) - {""})
    try:
        cnefe = leitura.carregar_cnefe(Path(opcoes.cnefe), municipios, duckdb.connect())
    except FileNotFoundError as erro:
        raise ErroDePipeline(str(erro)) from erro
    aviso(f"recorte do CNEFE: {len(cnefe):,} endereços em {len(municipios)} municípios")

    final = juntar(bruta, parear(busca, cnefe, opcoes))
    saida = escrita.salvar_resultado(final, Path(opcoes.saida))
    log = relatorio.montar_log(final, base_busca=caminho_busca.name,
                               municipios=municipios, enderecos_cnefe=len(cnefe),
                               segundos=time.time() - inicio)
    escrita.salvar_log(log, saida.parent)
    return Resultado(final=final, log=log, saida=saida, colunas_detectadas=mapa)


Overwriting src/laceda_setor/pipeline.py


#### A interface

O `cli.py` lê argumentos e imprime; `main.py` na raiz do `src/` é só um atalho de execução.


In [31]:
%%writefile src/laceda_setor/cli.py
"""
Interface de linha de comando. Responsabilidade única: ler argumentos, chamar o
pipeline e imprimir o resumo.

    python src/main.py                              # usa data/raw/
    python src/main.py --busca minha_base.xlsx      # outra base de busca
    python -m laceda_setor                          # idem, com PYTHONPATH=src
"""

from __future__ import annotations

import argparse
from pathlib import Path

from . import match as mt
from . import relatorio
from .config import CNEFE_PADRAO, SAIDA_PADRAO
from .pipeline import ErroDePipeline, Opcoes, executar


def analisar_argumentos(argv=None) -> Opcoes:
    ap = argparse.ArgumentParser(description="Localiza o setor censitário de cada endereço.")
    ap.add_argument("--cnefe", default=CNEFE_PADRAO, type=Path)
    ap.add_argument("--busca", default=None, type=Path, help="base de busca (.xlsx/.csv)")
    ap.add_argument("--saida", default=SAIDA_PADRAO, type=Path)
    ap.add_argument("--limiar-cep", type=float, default=mt.LIMIAR_FUZZY_CEP)
    ap.add_argument("--limiar-bairro", type=float, default=mt.LIMIAR_FUZZY_BAIRRO)
    args = ap.parse_args(argv)
    return Opcoes(cnefe=args.cnefe, busca=args.busca, saida=args.saida,
                  limiar_cep=args.limiar_cep, limiar_bairro=args.limiar_bairro)


def imprimir_resumo(resultado) -> None:
    print(f"\n{resultado.saida} — {len(resultado.final)} linhas")
    for linha in relatorio.linhas_resumo(resultado.final):
        print(linha)
    print(f"concluído em {resultado.log['segundos']}s")


def main(argv=None) -> None:
    try:
        resultado = executar(analisar_argumentos(argv), aviso=print)
    except ErroDePipeline as erro:
        raise SystemExit(str(erro))
    imprimir_resumo(resultado)


if __name__ == "__main__":
    main()


Overwriting src/laceda_setor/cli.py


In [32]:
%%writefile src/laceda_setor/__init__.py
"""
Localização do setor censitário de endereços no CNEFE (case CIDACS/LACEDA).

Um módulo por responsabilidade:
    config     — constantes e caminhos padrão
    leitura    — achar e ler CNEFE e base de busca
    colunas    — descobrir o esquema da base de busca
    normalize  — limpeza e padronização dos endereços (passo 2)
    match      — busca em cascata do setor censitário (passo 3)
    pipeline   — ordem dos passos, de ponta a ponta
    relatorio  — agregação dos números da execução
    escrita    — gravação do resultado e do log
    cli        — argumentos de linha de comando e impressão
    base_demo  — base de busca sintética, enquanto a real não chega
"""

from . import match, normalize, pipeline
from .pipeline import Opcoes, Resultado, executar

__all__ = ["match", "normalize", "pipeline", "executar", "Opcoes", "Resultado"]


Overwriting src/laceda_setor/__init__.py


In [33]:
%%writefile src/laceda_setor/__main__.py
"""Permite `python -m laceda_setor` (equivalente a `python src/main.py`)."""

from .cli import main

if __name__ == "__main__":
    main()


Overwriting src/laceda_setor/__main__.py


In [34]:
%%writefile src/main.py
"""
Atalho de execução: `python src/main.py [opções]`.

A implementação vive em `src/laceda_setor/cli.py`; este arquivo só coloca o
diretório `src/` no caminho de importação e delega.
"""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from laceda_setor.cli import main  # noqa: E402

if __name__ == "__main__":
    main()


Overwriting src/main.py


### 5.3 Testes de `normalize.py` com os casos do enunciado

Escreve a bateria de testes da normalização, cobrindo os exemplos do enunciado e cada achado
do Passo 1 que virou regra. Fixar isso em teste é o que impede uma regra de ser quebrada sem
ninguém notar.

O `conftest.py` coloca `src/` no caminho de importação uma vez só, para todos os testes:


In [35]:
%%writefile tests/conftest.py
"""Coloca `src/` no caminho de importação para os testes acharem `laceda_setor`."""

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))


Overwriting tests/conftest.py


In [36]:
%%writefile tests/test_normalize.py
"""Testes da normalização — os casos vêm do enunciado e dos achados do Passo 1."""

import pytest

from laceda_setor import normalize as nz


@pytest.mark.parametrize("entrada,esperado", [
    ("R. Rio Baiano", "RUA RIO BAIANO"),            # caso do enunciado
    ("RUA RIO BAIANO", "RUA RIO BAIANO"),
    ("Rodovia BR 101", "RODOVIA BR 101"),           # caso do enunciado
    ("ROD BR 101", "RODOVIA BR 101"),
    ("AV DR ABELARDO VELOSO", "AVENIDA DOUTOR ABELARDO VELOSO"),
    ("Pça N Sra da Conceição", "PRACA NOSSA SENHORA DA CONCEICAO"),
    ("estr. do coco", "ESTRADA DO COCO"),
    ("Faz. Boa Vista", "FAZENDA BOA VISTA"),        # tipo rural: 665 mil linhas no CNEFE
    ("POV. SAO JOSE", "POVOADO SAO JOSE"),
    ("  rua   das   flores  ", "RUA DAS FLORES"),
    ("", ""),
    (None, ""),
])
def test_logradouro(entrada, esperado):
    assert nz.normalizar_logradouro(entrada) == esperado


@pytest.mark.parametrize("entrada,esperado", [
    ("48030-260", "48030260"),
    ("48030260", "48030260"),
    (48030260, "48030260"),
    ("8030260", "08030260"),      # zero à esquerda comido pelo Excel
    ("", ""),
    (None, ""),
    ("00000000", ""),             # CEP inválido
    ("123456789", ""),            # dígitos demais
])
def test_cep(entrada, esperado):
    assert nz.normalizar_cep(entrada) == esperado


@pytest.mark.parametrize("entrada", ["S/N", "SN", "s/n", "", None, 0, "0", "SEM NUMERO"])
def test_sem_numero(entrada):
    assert nz.normalizar_numero(entrada) == "SN"


@pytest.mark.parametrize("entrada,esperado", [
    ("123", "123"), ("0045", "45"), ("123-A", "123"), ("123 FUNDOS", "123"),
])
def test_numero(entrada, esperado):
    assert nz.normalizar_numero(entrada) == esperado


def test_municipio_concilia_6_e_7_digitos():
    """O achado central do Passo 1: CNEFE usa 7 dígitos, a base de busca usa 6."""
    assert nz.normalizar_municipio("2900702") == nz.normalizar_municipio("290070") == "290070"
    assert nz.normalizar_municipio("2901304") == nz.normalizar_municipio("290130") == "290130"


def test_setor_perde_o_sufixo_P():
    """100% das 9 milhões de linhas do CNEFE têm 16 chars terminando em P."""
    assert nz.normalizar_setor("290070220000001P") == "290070220000001"
    assert len(nz.normalizar_setor("290070220000001P")) == 15


def test_composicao_do_logradouro_do_cnefe():
    """O CNEFE guarda o logradouro em 3 colunas; a base de busca, em uma só."""
    assert (nz.compor_logradouro_cnefe("AVENIDA", "DOUTOR", "ABELARDO VELOSO")
            == "AVENIDA DOUTOR ABELARDO VELOSO")
    assert nz.compor_logradouro_cnefe("RUA", "", "DAS FLORES") == "RUA DAS FLORES"
    assert nz.compor_logradouro_cnefe("RUA", None, "DAS FLORES") == "RUA DAS FLORES"


def test_os_dois_lados_convergem():
    """A regra de ouro: o mesmo endereço escrito de formas diferentes colapsa."""
    lado_busca = nz.normalizar_logradouro("Av. Dr. Abelardo Veloso")
    lado_cnefe = nz.compor_logradouro_cnefe("AVENIDA", "DOUTOR", "ABELARDO VELOSO")
    assert lado_busca == lado_cnefe


Overwriting tests/test_normalize.py


### 5.4 Testes da cascata de busca

Escreve os testes da busca sobre um CNEFE minúsculo montado à mão, com um caso para cada
nível da cascata. Os dois casos centrais são a distinção entre resposta resolvida e ambígua.

In [37]:
%%writefile tests/test_match.py
"""Testes da cascata — com um CNEFE minúsculo, montado à mão."""

import pandas as pd
import pytest

from laceda_setor import match as mt


def _cnefe(linhas):
    return pd.DataFrame(linhas, columns=["n_municipio", "n_cep", "n_logradouro",
                                         "n_numero", "n_bairro", "setor_15"])


@pytest.fixture
def indice():
    return mt.IndiceCNEFE(_cnefe([
        ("290070", "48030260", "RUA RIO BAIANO", "100", "CENTRO", "290070205000001"),
        ("290070", "48030260", "RUA RIO BAIANO", "102", "CENTRO", "290070205000001"),
        # mesmo logradouro, outro CEP, OUTRO setor -> fonte de ambiguidade sem CEP
        ("290070", "48031000", "RUA RIO BAIANO", "100", "KENNEDY", "290070205000099"),
        ("290070", "48030999", "AVENIDA DOUTOR ABELARDO VELOSO", "SN", "CENTRO",
         "290070205000002"),
        ("290130", "46830000", "POVOADO SAO JOSE", "SN", "ZONA RURAL", "290130205000003"),
    ]))


def test_nivel_1_exato_com_numero(indice):
    r = mt.buscar_setor({"n_municipio": "290070", "n_cep": "48030260",
                         "n_logradouro": "RUA RIO BAIANO", "n_numero": "100"}, indice)
    assert r["status"] == mt.STATUS_ENCONTRADO
    assert r["setor_censitario_encontrado"] == "290070205000001"
    assert r["regra"].startswith("1_")


def test_nivel_2_sem_numero(indice):
    """29% do CNEFE não tem número: precisa existir caminho que o ignore."""
    r = mt.buscar_setor({"n_municipio": "290070", "n_cep": "48030260",
                         "n_logradouro": "RUA RIO BAIANO", "n_numero": "SN"}, indice)
    assert r["status"] == mt.STATUS_ENCONTRADO
    assert r["regra"].startswith("2_")


def test_numeros_diferentes_mesmo_setor_nao_e_ambiguidade(indice):
    """Dois endereços distintos apontando para o MESMO setor: resposta resolvida."""
    r = mt.buscar_setor({"n_municipio": "290070", "n_cep": "48030260",
                         "n_logradouro": "RUA RIO BAIANO", "n_numero": "SN"}, indice)
    assert r["n_setores_candidatos"] == 1


def test_setores_diferentes_viram_ambiguo(indice):
    """Sem CEP, 'RUA RIO BAIANO 100' existe em dois setores -> não chutar."""
    r = mt.buscar_setor({"n_municipio": "290070", "n_cep": "",
                         "n_logradouro": "RUA RIO BAIANO", "n_numero": "100"}, indice)
    assert r["status"] == mt.STATUS_AMBIGUO
    assert r["setor_censitario_encontrado"] == ""
    assert r["n_setores_candidatos"] == 2


def test_nivel_4_fuzzy_dentro_do_cep(indice):
    """Letra faltando no logradouro, mas o CEP ancora a busca."""
    r = mt.buscar_setor({"n_municipio": "290070", "n_cep": "48030260",
                         "n_logradouro": "RUA RIO BAIANNO", "n_numero": "SN"}, indice)
    assert r["status"] == mt.STATUS_ENCONTRADO
    assert r["regra"].startswith("4_")
    assert r["score"] >= mt.LIMIAR_FUZZY_CEP


def test_nivel_5_fuzzy_por_bairro(indice):
    """Sem CEP, o bairro ainda oferece um bloco de busca."""
    r = mt.buscar_setor({"n_municipio": "290130", "n_cep": "",
                         "n_logradouro": "POVOADO SAO JOZE", "n_numero": "SN",
                         "n_bairro": "ZONA RURAL"}, indice)
    assert r["status"] == mt.STATUS_ENCONTRADO
    assert r["regra"].startswith("5_")
    assert r["confianca"] == "BAIXA"


def test_nivel_6_nao_encontrado(indice):
    r = mt.buscar_setor({"n_municipio": "290070", "n_cep": "99999999",
                         "n_logradouro": "RUA QUE NAO EXISTE", "n_numero": "1"}, indice)
    assert r["status"] == mt.STATUS_NAO_ENCONTRADO
    assert r["setor_censitario_encontrado"] == ""


def test_municipio_errado_nao_vaza_resposta(indice):
    """O blocking por município tem de ser estrito."""
    r = mt.buscar_setor({"n_municipio": "290130", "n_cep": "48030260",
                         "n_logradouro": "RUA RIO BAIANO", "n_numero": "100"}, indice)
    assert r["status"] == mt.STATUS_NAO_ENCONTRADO


def test_campos_insuficientes(indice):
    r = mt.buscar_setor({"n_municipio": "", "n_logradouro": ""}, indice)
    assert r["status"] == mt.STATUS_NAO_ENCONTRADO
    assert r["regra"].startswith("0_")


Overwriting tests/test_match.py


### 5.4b Testes da orquestração

Esquema da base de busca, junção das colunas e escolha do arquivo de entrada:


In [38]:
%%writefile tests/test_pipeline.py
"""Testes das peças de orquestração: esquema, junção e escolha da base de busca."""

import pandas as pd
import pytest

from laceda_setor import colunas as cl
from laceda_setor import pipeline as pl


def test_mapeia_colunas_do_enunciado():
    df = pd.DataFrame(columns=["id_consulta", "consulta_municipio", "consulta_cep",
                               "consulta_logradouro", "consulta_numero", "consulta_bairro"])
    assert cl.mapear_colunas(df) == {
        "municipio": "consulta_municipio",
        "cep": "consulta_cep",
        "logradouro": "consulta_logradouro",
        "numero": "consulta_numero",
        "bairro": "consulta_bairro",
    }
    assert cl.papeis_faltando(cl.mapear_colunas(df)) == set()


def test_acusa_papel_essencial_ausente():
    df = pd.DataFrame(columns=["municipio_ibge", "bairro"])
    assert cl.papeis_faltando(cl.mapear_colunas(df)) == {"cep", "logradouro"}


def test_preparar_busca_sem_logradouro_falha():
    df = pd.DataFrame({"cod_municipio": ["290070"], "cep": ["40000000"]})
    with pytest.raises(pl.ErroDePipeline):
        pl.preparar_busca(df)


def test_juntar_descarta_colunas_de_gabarito():
    bruta = pd.DataFrame({"id": [1], "consulta_cep": ["40000000"], "_gabarito_setor": ["x"]})
    respostas = pd.DataFrame({"status": ["ENCONTRADO"]})
    assert list(pl.juntar(bruta, respostas).columns) == ["id", "consulta_cep", "status"]


def test_resolver_base_busca_ignora_o_cnefe(tmp_path):
    (tmp_path / "29_BA.csv").touch()
    (tmp_path / "base_busca.xlsx").touch()
    assert pl.resolver_base_busca(None, tmp_path).name == "base_busca.xlsx"


def test_resolver_base_busca_sem_arquivo(tmp_path):
    with pytest.raises(pl.ErroDePipeline):
        pl.resolver_base_busca(None, tmp_path)


Overwriting tests/test_pipeline.py


### 5.5 Rodar os testes

Executa a bateria inteira. É a verificação de que os módulos recém-escritos pelo notebook
realmente funcionam — não bastaria tê-los gravado em disco.

In [39]:
!"{sys.executable}" -m pytest tests/ -q 2>&1 | tail -20

...................................................                      [100%]
51 passed in 0.49s


### 5.6 `Dockerfile`

Gera a receita de ambiente. `data/` e `output/` entram por volume, porque as bases não são
versionadas e o resultado precisa sair do contêiner.

In [40]:
%%writefile Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY tests/ ./tests/

# data/ e output/ entram por volume: as bases não são versionadas
#   docker build -t cidacs-setor .
#   docker run --rm -v "$PWD/data:/app/data" -v "$PWD/output:/app/output" cidacs-setor
ENV PYTHONPATH=/app/src
CMD ["python", "src/main.py"]

Overwriting Dockerfile


### 5.7 `README.md`

Escreve a documentação da entrega: abordagem, passo a passo de execução, dependências,
premissas e limitações. Os dois últimos itens são cobrados explicitamente pelo enunciado.

In [41]:
%%writefile README.md
# Localização de setor censitário a partir de endereços — CNEFE Bahia

Dado um conjunto de endereços com qualidade variável, encontra na base CNEFE da Bahia
(9.047.296 endereços) o **setor censitário** correspondente a cada um, distinguindo
explicitamente os casos encontrados, ambíguos e não encontrados.

## Abordagem

Pareamento determinístico em cascata, com recurso a similaridade textual quando a escrita não
bate exatamente. Três etapas:

1. **Normalização idêntica dos dois lados.** O mesmo código limpa o CNEFE e a base de busca:
   maiúsculas sem acento nem pontuação, abreviações expandidas (`R.`→`RUA`, `AV`→`AVENIDA`,
   `DR`→`DOUTOR`, e também os tipos rurais `FAZ`→`FAZENDA`, `POV`→`POVOADO`), CEP reduzido a 8
   dígitos com zero à esquerda, `S/N`/vazio/`0` colapsados em `SN`. Limpar os dois lados de
   formas diferentes é o erro que impede qualquer match.

2. **Blocking por município.** Só os municípios presentes na base de busca são carregados do
   CNEFE. Reduz o espaço de comparação em ordens de grandeza e dispensa Spark.

   O CSV de 1,46 GB é lido direto, uma vez por execução, sem camada intermediária em disco.
   Converter para Parquet economizaria ~9 s por execução e custaria ~12 s de conversão: não se
   paga, e criaria um cache a manter. O recorte (~1,5 milhão de endereços) segue em memória
   daí em diante.

3. **Busca em cascata**, do mais específico ao mais tolerante, parando no primeiro acerto:

   | nível | chave | confiança |
   |---|---|---|
   | 1 | município + CEP + logradouro + número (exato) | ALTA |
   | 2 | município + CEP + logradouro (exato) | ALTA |
   | 3 | município + logradouro + número (exato) | MEDIA |
   | 4 | município + CEP, logradouro parecido (`rapidfuzz` ≥ 88) | MEDIA |
   | 5 | município + bairro, logradouro parecido (≥ 92) | BAIXA |
   | 6 | nada bateu → `NAO_ENCONTRADO` | — |

   Quando vários endereços candidatos apontam para o **mesmo** setor, a resposta está resolvida.
   Quando apontam para setores **diferentes**, o status é `AMBIGUO` e o setor fica vazio — os
   candidatos ficam registrados em `setores_candidatos`, para auditoria.

A cascata existe porque o poder discriminante do CEP varia muito: 60% dos CEPs da Bahia cobrem
um único setor, mas há CEPs genéricos de município pequeno que cobrem dezenas (`46830000`,
Andaraí: 8.642 endereços em 38 setores). Nenhuma regra única resolve os dois casos.

## Como executar

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

Coloque as duas bases em `data/raw/` (não são versionadas):

| arquivo | origem |
|---|---|
| `29_BA.csv` | CNEFE BA — `ftp.ibge.gov.br` → Censo 2022 → Arquivos_CNEFE → CSV → UF → `29_BA.zip` |
| `base_busca.xlsx` | SharePoint da Fiocruz (link no enunciado; **exige login**) |

Depois:

```bash
python src/main.py
```

Roda em menos de um minuto. Saída em `output/resultado_final.csv` (+ `.parquet`) e
`output/log_execucao.json`.

Opções: `--busca OUTRA_BASE.xlsx`, `--saida caminho.csv`, `--limiar-cep 85`, `--limiar-bairro 95`.

### Docker

```bash
docker build -t cidacs-setor .
docker run --rm -v "$PWD/data:/app/data" -v "$PWD/output:/app/output" cidacs-setor
```

### Testes

```bash
python -m pytest tests/ -q
```

## Saída

As colunas originais da base de busca, intactas, mais:

| coluna | conteúdo |
|---|---|
| `setor_censitario_encontrado` | código de 15 dígitos; **vazio** quando não resolvido |
| `status` | `ENCONTRADO` / `AMBIGUO` / `NAO_ENCONTRADO` |
| `regra` | qual dos 6 níveis resolveu |
| `score` | similaridade (100 nos níveis exatos) |
| `confianca` | ALTA / MEDIA / BAIXA |
| `n_setores_candidatos`, `setores_candidatos` | o que a busca viu nos casos ambíguos |

## Estrutura

```
├── src/
│   ├── main.py                # atalho: `python src/main.py`
│   └── laceda_setor/          # o pacote — um módulo por responsabilidade
│       ├── config.py          # constantes e caminhos padrão
│       ├── leitura.py         # achar e ler CNEFE e base de busca
│       ├── colunas.py         # descobrir o esquema da base de busca
│       ├── normalize.py       # limpeza, aplicada igual aos dois lados
│       ├── match.py           # cascata de 6 níveis
│       ├── pipeline.py        # ordem dos passos, de ponta a ponta
│       ├── relatorio.py       # números da execução
│       ├── escrita.py         # grava resultado e log
│       ├── cli.py             # argumentos e impressão
│       └── base_demo.py       # base sintética (enquanto a real não está disponível)
├── tests/                     # pytest
├── notebooks/
│   ├── 01_analise_exploratoria.ipynb
│   └── 02_pipeline_plano_simples.ipynb
├── data/raw/          # bases de entrada (não versionadas)
└── output/            # resultado + logs
```

## Premissas

- **Município é bloco obrigatório.** Nenhuma resposta é dada fora do município informado na
  consulta. Se o município vier errado na entrada, o endereço fica `NAO_ENCONTRADO`.
- **O código de município da base de busca tem 6 dígitos e o do CNEFE tem 7** (o 7º é o
  verificador). A junção é feita pelos 6 primeiros.
- **`COD_SETOR` do CNEFE é cortado em 15 dígitos**, removendo o sufixo `P` presente em 100%
  das linhas, para corresponder ao formato do enunciado.
- **Ambiguidade não é resolvida por desempate.** Preferiu-se devolver vazio com os candidatos
  registrados a escolher um setor por critério arbitrário (frequência, primeiro encontrado).
- **Todos os campos são lidos como texto**, porque zeros à esquerda em CEP, setor e município
  são significativos.
- **O exemplo de saída do enunciado é ilustrativo.** Os setores `292740805000123` e
  `291230605000456` não existem no CNEFE (`2912306` nem é município válido da Bahia) — o
  exemplo define o formato, não um gabarito.

## Limitações conhecidas

- **Sem geocodificação.** O CNEFE traz latitude e longitude, mas a solução não usa busca
  espacial. Um endereço cuja grafia divergiu demais não é recuperado por proximidade
  geográfica, ainda que o ponto estivesse disponível.
- **Depende do município estar correto.** Não há tentativa de inferir o município a partir do
  CEP quando os dois se contradizem.
- **Limiares de similaridade são fixos** (88 e 92), calibrados por inspeção e não por uma base
  rotulada. O efeito de alterá-los está medido na seção 3.4 do Notebook 2.
- **O fuzzy compara só o logradouro**, dentro de um bloco de CEP ou bairro. Não há pontuação
  combinada entre campos (por exemplo, logradouro parecido + número exato valendo mais).
- **Endereços fora do cadastro não têm como ser resolvidos.** O CNEFE é de 2022; endereços
  novos ou informais simplesmente não constam, e o resultado correto para eles é
  `NAO_ENCONTRADO`.
- **Apenas a Bahia.** O recorte carregado é `29_BA`; outros estados exigiriam baixar os
  arquivos correspondentes.
- **Validado contra base sintética.** Enquanto a base de busca real não esteve disponível, a
  acurácia foi medida contra endereços sorteados do próprio CNEFE com sujeira injetada — um
  cenário otimista, já que todo endereço existe por construção.


Overwriting README.md


### 5.8 Conferir o que foi empacotado

Confere que cada arquivo prometido existe mesmo em disco. Vale mais do que supor: o notebook
gerou esses arquivos ao longo da execução, e uma célula pulada passaria despercebida.

In [42]:
MODULOS = ["config", "colunas", "leitura", "normalize", "match",
           "pipeline", "relatorio", "escrita", "cli", "base_demo"]

for caminho in (["README.md", "requirements.txt", "Dockerfile", "src/main.py"]
                + [f"src/laceda_setor/{m}.py" for m in MODULOS]
                + ["tests/conftest.py", "tests/test_normalize.py", "tests/test_match.py",
                   "tests/test_pipeline.py",
                   "output/resultado_final.csv", "output/log_execucao.json"]):
    p = Path(caminho)
    marca = "✅" if p.exists() else "❌"
    tamanho = f"{p.stat().st_size / 1024:.1f} KB" if p.exists() else "—"
    print(f"{marca} {caminho:<35} {tamanho:>10}")

✅ README.md                               7.2 KB
✅ requirements.txt                        0.1 KB
✅ Dockerfile                              0.4 KB
✅ src/main.py                             0.4 KB
✅ src/laceda_setor/config.py              1.3 KB
✅ src/laceda_setor/colunas.py             0.9 KB
✅ src/laceda_setor/leitura.py             1.7 KB
✅ src/laceda_setor/normalize.py           8.7 KB
✅ src/laceda_setor/match.py               6.8 KB
✅ src/laceda_setor/pipeline.py            3.8 KB
✅ src/laceda_setor/relatorio.py           1.2 KB
✅ src/laceda_setor/escrita.py             0.9 KB
✅ src/laceda_setor/cli.py                 1.7 KB
✅ src/laceda_setor/base_demo.py           3.2 KB
✅ tests/conftest.py                       0.2 KB
✅ tests/test_normalize.py                 3.0 KB
✅ tests/test_match.py                     4.0 KB
✅ tests/test_pipeline.py                  1.7 KB
✅ output/resultado_final.csv             13.0 KB
✅ output/log_execucao.json                0.6 KB


### 5.9 Verificação final: o `main.py` reproduz o mesmo CSV?

Roda o pipeline pela linha de comando e compara o resultado com o do notebook. É a prova de
que a entrega é reproduzível fora daqui — que o avaliador obtém o mesmo CSV com um comando.

In [43]:
if USANDO_BASE_REAL:
    !"{sys.executable}" src/main.py --saida execucao-NB/resultado_main.csv 2>&1 | tail -12
    a = pd.read_csv("output/resultado_final.csv", dtype=str).fillna("")
    b = pd.read_csv("execucao-NB/resultado_main.csv", dtype=str).fillna("")
    iguais = a["setor_censitario_encontrado"].equals(b["setor_censitario_encontrado"])
    print("\nnotebook e main.py produzem o mesmo setor em todas as linhas:", iguais)
else:
    print("pulado: main.py exige a base de busca real em data/raw/")
    print("o notebook usou a base sintética, que o main.py deliberadamente não gera")

pulado: main.py exige a base de busca real em data/raw/
o notebook usou a base sintética, que o main.py deliberadamente não gera


---
## Estado da entrega

| Passo | Estado |
|---|---|
| 1 — Olhar os dados | ✅ Notebook 1 + seção 1 |
| 2 — Limpar os endereços | ✅ `src/laceda_setor/normalize.py` + testes |
| 3 — Procurar o setor | ✅ `src/laceda_setor/match.py`, cascata de 6 níveis + testes |
| 4 — Gerar a saída | ✅ `output/resultado_final.csv` / `.parquet` / `.xlsx` + logs |
| 5 — Empacotar | ✅ README, requirements, Dockerfile, `main.py`, pytest |

**Pendência única:** a base de busca real (`data/raw/base_busca.xlsx`). Baixada e colocada ali,
tanto este notebook quanto `python src/main.py` passam a usá-la sem nenhuma alteração de código.

Os dois pontos do plano a confirmar com a comissão continuam de pé: o prazo do PDF
(18/05/2026) já passou, e os links do SharePoint exigem login.